<a href="https://colab.research.google.com/github/NickWu97/Music-Basics-Assessment/blob/main/sdxl_v1.0_comfyui_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 先強制修正套件版本，解決 no_init_weights 的報錯
!pip install transformers==4.35.0 --force-reinstall

# 2. 為了確保萬一，補裝一個穩定的加速套件
!pip install xformers==0.0.22.post7 --index-url https://download.pytorch.org/whl/cu118

# 3. 重新啟動 ComfyUI 服務 (這裡會自動產生新的連結)
%cd /content/ComfyUI
!python main.py --dont-print-server

In [ ]:
import os
import time
import subprocess
from threading import Timer
from queue import Queue
from IPython.display import clear_output

# --- 1. 清理環境並安裝核心依賴 ---
print("⚙️ 正在優化系統環境...")
!apt -y update -qq && apt -y install -qq aria2
!pip install -q torch==2.0.1+cu118 torchvision==0.15.2+cu118 --extra-index-url https://download.pytorch.org/whl/cu118
!pip install -q xformers==0.0.20 triton==2.0.0
# 關鍵：解決 AttributeError 的核心步驟
!pip install -q transformers==4.35.0 accelerate==0.21.0 --force-reinstall
!pip install -q mediapipe==0.9.1.0 addict yapf fvcore omegaconf

# --- 2. 安裝 ComfyUI 主程式 ---
%cd /content
if not os.path.exists("ComfyUI"):
    !git clone -b v1.3 https://github.com/camenduru/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt

# --- 3. 下載 AI 大腦 (SDXL 模型) ---
print("📦 正在搬運 AI 模型 (這需要約 5-8 分鐘，請保持連線)...")
# Base 模型
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "https://huggingface.co/ckpt/sd_xl_base_1.0/resolve/main/sd_xl_base_1.0_0.9vae.safetensors" -d /content/ComfyUI/models/checkpoints -o sd_xl_base_1.0.safetensors
# VAE 色彩修正
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "https://huggingface.co/ckpt/sdxl_vae/resolve/main/sdxl_vae.safetensors" -d /content/ComfyUI/models/vae -o sdxl_vae.vae.safetensors

# --- 4. 啟動 Cloudflare 安全隧道 ---
print("🌐 正在建立加密隧道...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared-linux-amd64 && chmod 777 /content/cloudflared-linux-amd64

def get_tunnel_url(port, metrics_port):
    import re, requests
    subprocess.Popen(['/content/cloudflared-linux-amd64', 'tunnel', '--url', f'http://127.0.0.1:{port}', '--metrics', f'127.0.0.1:{metrics_port}'], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    time.sleep(10)
    try:
        metrics = requests.get(f'http://127.0.0.1:{metrics_port}/metrics').text
        url = re.search("(?P<url>https?:\/\/[^\s]+.trycloudflare.com)", metrics).group("url")
        return url
    except:
        return None

# --- 5. 正式啟動服務 ---
url = get_tunnel_url(8188, 8100)
clear_output()

if url:
    print(f"✅ 環境部署成功！")
    print(f"🔗 你的 AI 控制台連結: {url}")
    print(f"💡 貼心提醒：進去後請等待方塊載入，然後直接點『Queue Prompt』測試。")
    # 啟動後端並鎖定畫面
    !python main.py --dont-print-server
else:
    print("❌ 隧道建立失敗，請重按一次播放鍵。")